# Análise dos resultados — viga soldada

Este notebook analisa as **35 execuções independentes** da Evolução Diferencial
`DE/rand/1/bin` autoadaptativa. Cada linha do CSV de resultados já contém o melhor
indivíduo devolvido por uma execução. A análise mantém uma linha por semente,
seleciona o melhor global entre as soluções viáveis e produz comparações com
`APM Med 3` e com a linha `Este estudo` da dissertação de Érica Carvalho.

Como o problema é de minimização, valores menores são melhores. Soluções viáveis
são ordenadas pelo objetivo original; a aptidão penalizada só é usada como fallback
se uma configuração não produzir nenhuma solução viável.


## 1. Leitura e seleção das 35 execuções

Arquivos antigos podem conter repetições das mesmas sementes. A configuração
experimental atual é identificada pelas colunas do algoritmo, população, dimensão,
orçamento, passo de registro e peso de penalização. Dentro dela, duplicatas de uma
mesma semente são removidas mantendo o registro mais recente. O notebook interrompe
a execução se não encontrar exatamente uma configuração com as sementes 1 a 35.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

EXPECTED_RUNS = 35
PROBLEM_DIRECTORY = 'Probem_viga'
candidate = Path.cwd()
BASE_DIR = (
    candidate
    if (candidate / 'results').is_dir()
    else candidate / 'work_3' / PROBLEM_DIRECTORY
)
if not (BASE_DIR / 'results').is_dir():
    raise FileNotFoundError(
        'Execute o notebook a partir de sua pasta ou da raiz do repositório.'
    )

RESULTS_DIR = BASE_DIR / 'results'
EXPORT_DIR = BASE_DIR / 'analysis_exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

result_files = sorted(RESULTS_DIR.glob('*.csv'))
if not result_files:
    raise FileNotFoundError(f'Nenhum resultado CSV encontrado em {RESULTS_DIR}')

raw_results = pd.concat(
    (pd.read_csv(path) for path in result_files), ignore_index=True
)
required_columns = {
    'algorithm', 'population_size', 'dimension',
    'max_fitness_evaluations', 'evaluation_step', 'penalty_weight',
    'seed', 'best_f', 'best_objective', 'penalty',
    'max_constraint_violation', 'is_feasible',
    'best_x_0', 'best_x_1', 'best_x_2', 'best_x_3'
}
missing_columns = required_columns.difference(raw_results.columns)
if missing_columns:
    raise ValueError(f'Colunas ausentes nos resultados: {sorted(missing_columns)}')

raw_results['is_feasible'] = (
    raw_results['is_feasible'].astype(str).str.lower()
    .map({'true': True, 'false': False})
)
if raw_results['is_feasible'].isna().any():
    raise ValueError('A coluna is_feasible contém valores inválidos.')

configuration_columns = [
    'algorithm', 'population_size', 'dimension',
    'max_fitness_evaluations', 'evaluation_step', 'penalty_weight',
]
deduplicated = raw_results.drop_duplicates(
    configuration_columns + ['seed'], keep='last'
)
configuration_counts = (
    deduplicated.groupby(configuration_columns, dropna=False)['seed']
    .nunique().rename('numero_de_sementes').reset_index()
)
candidates = configuration_counts[
    configuration_counts['numero_de_sementes'].eq(EXPECTED_RUNS)
]
if len(candidates) != 1:
    raise ValueError(
        'Era esperada exatamente uma configuração com 35 sementes; '
        f'foram encontradas {len(candidates)}. Configurações: '
        f'{configuration_counts}'
    )

selected_configuration = candidates.iloc[0]
configuration_mask = pd.Series(True, index=deduplicated.index)
for column in configuration_columns:
    configuration_mask &= deduplicated[column].eq(
        selected_configuration[column]
    )

best_per_run = (
    deduplicated.loc[configuration_mask]
    .sort_values('seed').reset_index(drop=True)
)
expected_seeds = set(range(1, EXPECTED_RUNS + 1))
actual_seeds = set(best_per_run['seed'].astype(int))
if actual_seeds != expected_seeds:
    raise ValueError(
        f'Sementes inesperadas. Esperado: {sorted(expected_seeds)}; '
        f'obtido: {sorted(actual_seeds)}.'
    )

print(
    f'{len(raw_results)} linhas lidas; '
    f'{len(best_per_run)} execuções únicas selecionadas (seeds 1–35).'
)
display(selected_configuration[configuration_columns].to_frame('valor'))


39 linhas lidas; 35 execuções únicas selecionadas (seeds 1–35).


,valor
algorithm,Self-adaptive DE/rand/1/bin with quadratic sta...
population_size,30
dimension,4
max_fitness_evaluations,320000
evaluation_step,30
penalty_weight,10000.0


## 2. Melhor indivíduo de cada execução

A tabela abaixo é a base auditável das estatísticas. Ela contém o melhor indivíduo
de cada uma das 35 execuções, sua viabilidade, objetivo original, aptidão penalizada,
penalidade, violação máxima e variáveis de projeto.


In [2]:
variable_map = {'h': 'best_x_0', 'l': 'best_x_1', 't': 'best_x_2', 'b': 'best_x_3'}
per_run_columns = [
    'seed', 'best_objective', 'best_f', 'penalty',
    'max_constraint_violation', 'is_feasible',
] + list(variable_map.values())
per_run_rename = {
    'best_objective': 'custo',
    'best_f': 'aptidao_penalizada',
    'max_constraint_violation': 'violacao_maxima',
    'is_feasible': 'viavel',
    **{source: target for target, source in variable_map.items()},
}
best_per_run_table = (
    best_per_run[per_run_columns].rename(columns=per_run_rename)
)
best_per_run_path = EXPORT_DIR / 'melhores_por_execucao.csv'
best_per_run_table.to_csv(
    best_per_run_path, index=False, float_format='%.12g'
)
display(best_per_run_table)
print(f'CSV exportado: {best_per_run_path}')


,seed,custo,aptidao_penalizada,penalty,violacao_maxima,viavel,h,l,t,b
0,1,2.381136,2.381136,0.0,-0.0,True,0.244368,6.218603,8.291497,0.244369
1,2,2.381152,2.381152,0.0,0.0,True,0.244366,6.218323,8.291875,0.244366
2,3,2.381134,2.381134,0.0,0.0,True,0.244369,6.218601,8.291480,0.244369
3,4,2.381163,2.381163,0.0,0.0,True,0.244357,6.219031,8.291472,0.244369
4,5,2.381145,2.381145,0.0,0.0,True,0.244365,6.218733,8.291499,0.244369
5,6,2.382646,2.382646,0.0,0.0,True,0.244098,6.212285,8.308266,0.244302
6,7,2.381134,2.381134,0.0,0.0,True,0.244369,6.218603,8.291478,0.244369
7,8,2.381134,2.381134,0.0,-0.0,True,0.244369,6.218607,8.291472,0.244369
8,9,2.381193,2.381193,0.0,-0.0,True,0.244355,6.218276,8.292347,0.244363
9,10,2.381466,2.381466,0.0,0.0,True,0.244233,6.223425,8.291477,0.244369


CSV exportado: /home/esterci/Repositorios/ppgmc/Evolutionary_Algorithms/work_3/Probem_viga/analysis_exports/melhores_por_execucao.csv


## 3. Comparação com APM Med 3

As estatísticas deste trabalho são calculadas apenas sobre os melhores indivíduos
viáveis das 35 execuções: melhor, mediana, média, desvio-padrão **amostral** e pior.
Os valores de `APM Med 3` são transcritos da Tabela 5.16 da
[dissertação de Érica Carvalho](https://repositorio.ufjf.br/jspui/bitstream/ufjf/3506/1/ericadacostareiscarvalho.pdf). A coluna de soluções viáveis mantém
a notação apresentada na dissertação.


In [3]:
feasible_runs = best_per_run[best_per_run['is_feasible']]
if feasible_runs.empty:
    global_best = best_per_run.loc[best_per_run['best_f'].idxmin()]
    analyzed_values = best_per_run['best_f']
    selection_note = 'fallback pela menor aptidão penalizada'
else:
    global_best = feasible_runs.loc[feasible_runs['best_objective'].idxmin()]
    analyzed_values = feasible_runs['best_objective']
    selection_note = 'menor objetivo entre as soluções viáveis'

current_summary = {
    'metodo': 'DE/rand/1/bin autoadaptativo (este trabalho)',
    'melhor': analyzed_values.min(),
    'mediana': analyzed_values.median(),
    'media': analyzed_values.mean(),
    'desvio_padrao': analyzed_values.std(ddof=1),
    'pior': analyzed_values.max(),
    'solucoes_viaveis': f'{len(feasible_runs)}/{len(best_per_run)}',
}
apm_med_3_summary = {
    'metodo': 'APM Med 3 (dissertação)',
    **{'melhor': 2.38114, 'mediana': 2.43315, 'media': 2.67102, 'desvio_padrao': 2.0656, 'pior': 3.46638, 'solucoes_viaveis': '35/35'},
}
comparison_apm_med_3 = pd.DataFrame(
    [current_summary, apm_med_3_summary]
)
comparison_path = EXPORT_DIR / 'comparacao_apm_med_3.csv'
comparison_apm_med_3.to_csv(
    comparison_path, index=False, float_format='%.12g'
)

display(comparison_apm_med_3)
display(Markdown(
    f'**Melhor global:** seed {int(global_best["seed"])} '
    f'({selection_note}), custo = '
    f'{global_best["best_objective"]:.12g}, '
    f'viável = {bool(global_best["is_feasible"])}.'
))
print(f'CSV exportado: {comparison_path}')


,metodo,melhor,mediana,media,desvio_padrao,pior,solucoes_viaveis
0,DE/rand/1/bin autoadaptativo (este trabalho),2.381134,2.381145,2.39953,0.102588,2.988928,35/35
1,APM Med 3 (dissertação),2.381140,2.433150,2.67102,2.065600,3.466380,35/35


**Melhor global:** seed 28 (menor objetivo entre as soluções viáveis), custo = 2.38113411689, viável = True.

CSV exportado: /home/esterci/Repositorios/ppgmc/Evolutionary_Algorithms/work_3/Probem_viga/analysis_exports/comparacao_apm_med_3.csv


## 4. Variáveis de projeto: melhor global × “Este estudo”

O melhor global deste trabalho é contrastado com a linha `Este estudo` da
Tabela 5.17 da dissertação. A terceira linha é a diferença assinada
`este trabalho − dissertação`; portanto, valores negativos do objetivo favorecem este
trabalho em um problema de minimização.


In [4]:
variable_names = ['h', 'l', 't', 'b']
objective_label = 'custo'
current_design = {
    'fonte': 'Este trabalho (melhor global)',
    **{name: global_best[source] for name, source in variable_map.items()},
    objective_label: global_best['best_objective'],
}
study_design = {
    'fonte': 'Este estudo (dissertação)',
    **{'h': 0.2443, 'l': 6.2186, 't': 8.2914, 'b': 0.2443, 'custo': 2.3811},
}
difference_design = {
    'fonte': 'Diferença (este trabalho - dissertação)',
    **{
        column: current_design[column] - study_design[column]
        for column in variable_names + [objective_label]
    },
}
design_comparison = pd.DataFrame(
    [current_design, study_design, difference_design],
    columns=['fonte'] + variable_names + [objective_label],
)
design_path = EXPORT_DIR / 'variaveis_projeto_melhor_global.csv'
design_comparison.to_csv(
    design_path, index=False, float_format='%.12g'
)

display(design_comparison)
print(f'CSV exportado: {design_path}')


,fonte,h,l,t,b,custo
0,Este trabalho (melhor global),0.244369,6.218607,8.291472,0.244369,2.381134
1,Este estudo (dissertação),0.244300,6.218600,8.291400,0.244300,2.381100
2,Diferença (este trabalho - dissertação),0.000069,0.000007,0.000072,0.000069,0.000034


CSV exportado: /home/esterci/Repositorios/ppgmc/Evolutionary_Algorithms/work_3/Probem_viga/analysis_exports/variaveis_projeto_melhor_global.csv


## 5. Arquivos produzidos e observações

Os CSVs são gravados em `analysis_exports/`:

- `melhores_por_execucao.csv`: rastreabilidade das 35 sementes;
- `comparacao_apm_med_3.csv`: estatísticas deste trabalho e da dissertação;
- `variaveis_projeto_melhor_global.csv`: variáveis do melhor global, referência e diferenças.

Esta comparação é descritiva. Os métodos têm o mesmo orçamento e número de
execuções reportados na dissertação, mas não há dados pareados por semente do
`APM Med 3`; portanto, o notebook não aplica um teste estatístico pareado nem faz
inferências de superioridade apenas a partir do melhor caso.
